# Agentic Safety Probe - RunPod Experiment

Este notebook ejecuta el experimento completo en RunPod.

**Requisitos del Pod:**
- GPU: RTX 4090 (24GB) o superior
- Template: RunPod PyTorch 2.1+
- Disco: 50GB

**Tiempo estimado:** ~30-45 min para el experimento completo

## 1. Setup - Instalar dependencias

In [ ]:
# Verificar GPU disponible
import torch
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Instalar dependencias (solo la primera vez)
!pip install -q transformers>=4.40.0 accelerate>=0.27.0 bitsandbytes>=0.43.0
!pip install -q scikit-learn scipy numpy pandas matplotlib seaborn
!pip install -q tqdm jsonlines

In [ ]:
# Login a Hugging Face (necesario para Mistral-7B, es modelo gated)
# Obtén tu token en: https://huggingface.co/settings/tokens
# Acepta la licencia en: https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3
from huggingface_hub import login
login()  # Te pedirá el token interactivamente

## 2. Configuración del Experimento

In [ ]:
# ============================================
# CONFIGURACIÓN - Modifica aquí según necesites
# ============================================

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
N_SAMPLES = 20          # Muestras por categoría (reducir a 5 para test rápido)
INCLUDE_HISTORY = True  # Incluir historial multi-turno en formato agente
DECOMPOSE = True        # Activar análisis de descomposición (4 condiciones)
SEED = 42
OUTPUT_DIR = "results"

# Capas a analizar (None = todas cada 2 capas)
# Para test rápido usa: LAYERS = [10, 14, 18, 22]
LAYERS = None

# Pasos opcionales (poner False para ir más rápido)
RUN_PROBES = True
RUN_INTERVENTION = False  # Este es el paso más lento

N_PERMUTATIONS = 10000  # Para el test de significancia estadística

## 3. Cargar el Modelo

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import time
from pathlib import Path

np.random.seed(SEED)

from src.model_loader import (
    load_model_and_tokenizer,
    format_chat_prompt,
    format_agent_prompt,
    format_agent_prompt_with_history,
    format_role_only_prompt,
    format_role_plus_tools_prompt,
    get_model_info,
)

print("Cargando modelo (esto toma 2-5 minutos)...")
start = time.time()

model, tokenizer = load_model_and_tokenizer(
    model_name=MODEL_NAME,
    quantize=None,  # Auto-detect: FP16 si hay VRAM suficiente
)

model_info = get_model_info(model)
print(f"\nModelo cargado en {time.time() - start:.1f}s")
print(f"Capas: {model_info['num_layers']}")
print(f"Hidden dim: {model_info['hidden_size']}")
print(f"Memoria: {model_info['memory_gb']:.2f} GB")

## 4. Construir Dataset

In [ ]:
from data.build_dataset import build_dataset

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

dataset = build_dataset(
    output_path=f"{OUTPUT_DIR}/dataset.jsonl",
    n_harmful_per_category=N_SAMPLES,
    n_benign_per_category=N_SAMPLES,
    decompose=DECOMPOSE,
)

# Mostrar resumen
from collections import Counter
variants = Counter(d['variant'] for d in dataset)
print(f"Dataset total: {len(dataset)} prompts")
for v, count in sorted(variants.items()):
    print(f"  {v}: {count}")

## 5. Extraer Activaciones

Este es el paso más intensivo en GPU. Extrae las representaciones internas del modelo para cada prompt en formato chat y agente.

In [ ]:
from src.activation_extractor import ActivationExtractor
from src.model_loader import format_role_only_prompt, format_role_plus_tools_prompt

# Determinar capas
n_layers = model_info["num_layers"]
if LAYERS is None:
    layers = list(range(0, n_layers, 2))  # Cada 2 capas
else:
    layers = LAYERS

print(f"Analizando {len(layers)} capas: {layers}")

# Extractor
extractor = ActivationExtractor(model, tokenizer)

# Agrupar por variante (dinámico, soporta 2 o 4 formatos)
variant_names = sorted(set(d['variant'] for d in dataset))
variants_data = {v: [d for d in dataset if d["variant"] == v] for v in variant_names}
print(f"Variantes: {variant_names}")

all_activations = {}

for variant_name, entries in variants_data.items():
    print(f"\nProcesando {variant_name} ({len(entries)} prompts)...")
    variant_activations = {layer: [] for layer in layers}

    for i, entry in enumerate(entries):
        if (i + 1) % 5 == 0:
            print(f"  {i + 1}/{len(entries)}")

        fmt = entry["format"]
        if fmt == "chat":
            prompt = format_chat_prompt(tokenizer, entry["base_prompt"])
        elif fmt == "role_only":
            prompt = format_role_only_prompt(tokenizer, entry["base_prompt"])
        elif fmt == "role_plus_tools":
            prompt = format_role_plus_tools_prompt(tokenizer, entry["base_prompt"])
        elif fmt in ("agent_full", "agent"):
            if INCLUDE_HISTORY:
                prompt = format_agent_prompt_with_history(tokenizer, entry["base_prompt"])
            else:
                prompt = format_agent_prompt(tokenizer, entry["base_prompt"])
        else:
            prompt = format_chat_prompt(tokenizer, entry["base_prompt"])

        acts = extractor.extract(prompt, layers=layers)
        for layer in layers:
            variant_activations[layer].append(acts[layer])

    all_activations[variant_name] = {
        layer: np.stack(variant_activations[layer]) for layer in layers
    }
    print(f"  Shape: {all_activations[variant_name][layers[0]].shape}")

print("\n✓ Activaciones extraídas para todas las condiciones")

## 6. Análisis Principal: Dirección de Refusal + Gap Estadístico

Aquí se computa:
1. **Dirección de refusal** (difference-in-means)
2. **Validación PCA** (confirma que la dirección es el componente principal)
3. **ΔP con significancia estadística** (permutation test + Cohen's d)

In [ ]:
from src.refusal_direction import (
    compute_refusal_direction,
    compute_refusal_direction_pca,
    project_onto_direction,
    compute_projection_gap,
    full_refusal_analysis,
    compute_gap_by_layer,
)

print("=" * 60)
print("ANÁLISIS DE DIRECCIÓN DE REFUSAL")
print("=" * 60)

# Determinar la key del agente (backward compat)
agent_key = "agent_full_harmful" if "agent_full_harmful" in all_activations else "agent_harmful"

# Análisis completo por capa (chat vs agent_full)
layer_results = compute_gap_by_layer(
    harmful_activations={l: all_activations["chat_harmful"][l] for l in layers},
    benign_activations={l: all_activations["chat_benign"][l] for l in layers},
    chat_harmful_acts={l: all_activations["chat_harmful"][l] for l in layers},
    agent_harmful_acts={l: all_activations[agent_key][l] for l in layers},
    n_permutations=N_PERMUTATIONS,
    seed=SEED,
)

# Resumen por capa
print(f"\n{'Capa':<6} {'ΔP':<10} {'p-value':<12} {'Cohen d':<10} {'PCA cos':<10} {'Sig?'}")
print("-" * 60)
for layer in layers:
    r = layer_results[layer]
    gap = r['gap_analysis']
    pca = r['pca_validation']
    sig = "✓" if gap['is_significant_005'] else "✗"
    print(f"{layer:<6} {gap['delta_p']:<10.4f} {gap['p_value_permutation']:<12.6f} "
          f"{gap['cohens_d']:<10.3f} {pca['cosine_similarity_with_diff_means']:<10.3f} {sig}")

In [ ]:
# Encontrar la mejor capa (mayor gap significativo)
significant_layers = {
    l: r['gap_analysis']['delta_p'] 
    for l, r in layer_results.items() 
    if r['gap_analysis']['is_significant_005']
}

if significant_layers:
    best_layer = max(significant_layers, key=significant_layers.get)
    best = layer_results[best_layer]
    print(f"\n{'='*60}")
    print(f"MEJOR CAPA: {best_layer}")
    print(f"{'='*60}")
    print(f"  ΔP = {best['gap_analysis']['delta_p']:.4f}")
    print(f"  p-value (permutation) = {best['gap_analysis']['p_value_permutation']:.6f}")
    print(f"  p-value (Welch t-test) = {best['gap_analysis']['p_value_ttest_welch']:.6f}")
    print(f"  Cohen's d = {best['gap_analysis']['cohens_d']:.3f}")
    print(f"  PCA cosine similarity = {best['pca_validation']['cosine_similarity_with_diff_means']:.3f}")
    print(f"  PCA explained variance (PC1) = {best['pca_validation']['explained_variance_ratio_pc1']:.3f}")
    print(f"\nInterpretación:")
    d = abs(best['gap_analysis']['cohens_d'])
    effect = "grande" if d > 0.8 else "medio" if d > 0.5 else "pequeño"
    print(f"  Efecto: {effect} (d={d:.3f})")
    if best['gap_analysis']['delta_p'] > 0:
        print(f"  → El formato AGENTE REDUCE la proyección sobre la dirección de refusal")
        print(f"  → Esto sugiere que el contexto agéntico DEBILITA el mecanismo de rechazo")
    else:
        print(f"  → No se observa debilitamiento del refusal en formato agente")
else:
    print("\n⚠ No se encontraron capas con gap significativo (p < 0.05)")
    print("Esto podría indicar que el formato agente NO debilita el refusal,")
    print("o que se necesitan más muestras.")
    best_layer = layers[len(layers)//2]  # usar capa media como fallback

## 6b. Análisis de Descomposición (si DECOMPOSE=True)

Descompone el efecto total en 3 factores:
- **Role effect**: ¿Asumir rol de agente reduce refusal?
- **Tools effect**: ¿Tener herramientas en el contexto reduce refusal?
- **JSON format effect**: ¿Pedir output en formato JSON/tool-call reduce refusal?

In [ ]:
if DECOMPOSE:
    from src.metrics import decompose_format_effects

    # Computar dirección de refusal en la mejor capa
    direction = compute_refusal_direction(
        all_activations["chat_harmful"][best_layer],
        all_activations["chat_benign"][best_layer],
    )

    # Proyectar las 4 condiciones
    conditions_order = ["chat_harmful", "role_only_harmful",
                        "role_plus_tools_harmful", "agent_full_harmful"]

    decomp_projections = {}
    for cond in conditions_order:
        decomp_projections[cond] = project_onto_direction(
            all_activations[cond][best_layer], direction
        )

    # Descomponer
    decomp = decompose_format_effects(decomp_projections, n_permutations=N_PERMUTATIONS, seed=SEED)

    # Resultados
    print("=" * 70)
    print(f"DESCOMPOSICIÓN DEL EFECTO (Capa {best_layer})")
    print("=" * 70)
    print(f"\nProyecciones medias:")
    for cond in conditions_order:
        m = decomp_projections[cond].mean()
        s = decomp_projections[cond].std()
        print(f"  {cond:<30} = {m:.4f} (±{s:.4f})")

    print(f"\n{'Efecto':<25} {'ΔP':<10} {'p-value':<12} {'Cohen d':<10} {'CI 95%':<20} {'Sig?'}")
    print("-" * 80)
    for name in ["role_effect", "tools_effect", "json_format_effect", "total_effect"]:
        e = decomp[name]
        sig = "✓" if e["significant"] else "✗"
        ci = f"[{e['ci_lower']:.4f}, {e['ci_upper']:.4f}]"
        print(f"  {name:<23} {e['delta_p']:<10.4f} {e['p_value']:<12.6f} "
              f"{e['cohens_d']:<10.3f} {ci:<20} {sig}")

    # Porcentaje de contribución
    total = abs(decomp["total_effect"]["delta_p"])
    if total > 1e-10:
        print(f"\nContribución relativa al efecto total:")
        for name in ["role_effect", "tools_effect", "json_format_effect"]:
            pct = decomp[name]["delta_p"] / total * 100
            bar = "█" * int(abs(pct) / 2)
            print(f"  {name:<25} {pct:>6.1f}% {bar}")
else:
    print("Descomposición desactivada (DECOMPOSE=False)")

## 7. Visualizaciones

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Plot 1: ΔP por capa ---
gaps = [layer_results[l]['gap_analysis']['delta_p'] for l in layers]
pvals = [layer_results[l]['gap_analysis']['p_value_permutation'] for l in layers]
colors = ['#e74c3c' if p < 0.05 else '#95a5a6' for p in pvals]

axes[0].bar(range(len(layers)), gaps, color=colors, alpha=0.8)
axes[0].set_xticks(range(len(layers)))
axes[0].set_xticklabels(layers, rotation=45, fontsize=8)
axes[0].set_xlabel('Capa')
axes[0].set_ylabel('ΔP (chat - agent)')
axes[0].set_title('Projection Gap por Capa\n(rojo = p < 0.05)')
axes[0].axhline(0, color='black', linewidth=0.5)

# --- Plot 2: Distribuciones en mejor capa ---
best_gap = layer_results[best_layer]['gap_analysis']
sns.kdeplot(best_gap['chat_projections'], ax=axes[1], label='Chat harmful', color='#e74c3c')
sns.kdeplot(best_gap['agent_projections'], ax=axes[1], label='Agent harmful', color='#3498db')
axes[1].axvline(best_gap['mean_chat_harmful'], color='#e74c3c', linestyle='--', alpha=0.7)
axes[1].axvline(best_gap['mean_agent_harmful'], color='#3498db', linestyle='--', alpha=0.7)
axes[1].set_xlabel('Proyección sobre dirección de refusal')
axes[1].set_ylabel('Densidad')
axes[1].set_title(f'Distribuciones (Capa {best_layer})\nΔP={best_gap["delta_p"]:.4f}, p={best_gap["p_value_permutation"]:.4f}')
axes[1].legend()

# --- Plot 3: PCA validation ---
cos_sims = [layer_results[l]['pca_validation']['cosine_similarity_with_diff_means'] for l in layers]
axes[2].plot(range(len(layers)), cos_sims, 'o-', color='#2ecc71')
axes[2].axhline(0.8, color='red', linestyle='--', alpha=0.5, label='Umbral alineación (0.8)')
axes[2].set_xticks(range(len(layers)))
axes[2].set_xticklabels(layers, rotation=45, fontsize=8)
axes[2].set_xlabel('Capa')
axes[2].set_ylabel('Cosine Similarity (PCA vs Diff-means)')
axes[2].set_title('Validación PCA por Capa')
axes[2].legend()
axes[2].set_ylim(0, 1.05)

plt.tight_layout()
Path(f"{OUTPUT_DIR}/figures").mkdir(parents=True, exist_ok=True)
plt.savefig(f"{OUTPUT_DIR}/figures/main_results.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Figura guardada en {OUTPUT_DIR}/figures/main_results.png")

## 8. Guardar Resultados

In [ ]:
import json

# Preparar resultados serializables
results_json = {
    "model": MODEL_NAME,
    "n_samples": N_SAMPLES,
    "n_permutations": N_PERMUTATIONS,
    "include_history": INCLUDE_HISTORY,
    "best_layer": int(best_layer),
    "layers_analyzed": layers,
    "per_layer": {},
}

for layer in layers:
    r = layer_results[layer]
    results_json["per_layer"][str(layer)] = {
        "delta_p": r['gap_analysis']['delta_p'],
        "p_value_permutation": r['gap_analysis']['p_value_permutation'],
        "p_value_ttest": r['gap_analysis']['p_value_ttest_welch'],
        "cohens_d": r['gap_analysis']['cohens_d'],
        "is_significant_005": r['gap_analysis']['is_significant_005'],
        "pca_cosine_similarity": r['pca_validation']['cosine_similarity_with_diff_means'],
        "pca_explained_variance_pc1": r['pca_validation']['explained_variance_ratio_pc1'],
    }

with open(f"{OUTPUT_DIR}/results.json", "w") as f:
    json.dump(results_json, f, indent=2)

print(f"✓ Resultados guardados en {OUTPUT_DIR}/results.json")
print(f"\nResumen final:")
print(json.dumps(results_json, indent=2, default=str))

## 9. (Opcional) Descargar resultados

Ejecuta esta celda para comprimir los resultados y descargarlos desde JupyterLab.

In [ ]:
# Comprimir resultados para descargar
!tar -czf results.tar.gz results/
print("\n✓ Archivo results.tar.gz creado")
print("→ En JupyterLab: click derecho sobre results.tar.gz → Download")